# 04 — Аудиоэмбеддинги каталога

Выкачивает `embeddings.parquet` YAMBDA (13.8 ГБ) и оставляет строки нашего
каталога в `artifacts/audio/embeddings.npy`. Логика — в
`research/scripts/build_audio.py`.

Ниже — проверка гипотезы «items без аудио это длинный хвост».

In [ ]:
# Colab: раскомментировать. Локально ячейка не нужна.
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -q https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q uv && uv pip install --system -e ".[research]"


In [ ]:
CONFIG = 'research/configs/aggregators_50m.yaml'

!uv run python research/scripts/build_audio.py --config {CONFIG}


## Пропуски аудио: это хвост каталога?

Сравниваем popularity items с эмбеддингом и без.

In [ ]:
import pickle
from pathlib import Path

import numpy as np

from grouprec.data.yambda_loader import filter_listens, load_yambda

ARTIFACTS = Path.cwd() / 'artifacts'
with open(ARTIFACTS / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)

embeds = np.load(ARTIFACTS / 'audio' / 'embeddings.npy')
missing_idx = set((np.where(np.linalg.norm(embeds[1:], axis=1) == 0)[0] + 1).tolist())
idx_to_id = {v: k for k, v in item_id_to_idx.items()}

counts = filter_listens(load_yambda('50m')['interactions'])['item_id'].value_counts()
missing_pop = counts.reindex([idx_to_id[i] for i in missing_idx]).dropna()
present_pop = counts.reindex(
    [idx_to_id[i] for i in range(1, embeds.shape[0]) if i not in missing_idx]).dropna()

print(f'без аудио ({len(missing_pop):,}): медиана={missing_pop.median():.0f}, p90={missing_pop.quantile(0.9):.0f}')
print(f'с аудио   ({len(present_pop):,}): медиана={present_pop.median():.0f}, p90={present_pop.quantile(0.9):.0f}')
for thr in (5, 10, 50, 100):
    print(f'  доля без аудио с pop<{thr}: {(missing_pop < thr).mean():.1%}')
